In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import datetime

# Data Cleaning and Preparation

In [2]:
sold = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week3/sold_with_rates.csv')
listing = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week3/listings_with_rates.csv')

/var/folders/q8/sdky6zrj6y52rcdpgrsws4vm0000gn/T/ipykernel_12925/920196162.py:1: DtypeWarning: Columns (0,1,9,64,78,79,80,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week3/sold_with_rates.csv')
/var/folders/q8/sdky6zrj6y52rcdpgrsws4vm0000gn/T/ipykernel_12925/920196162.py:2: DtypeWarning: Columns (2,43) have mixed types. Specify dtype option on import or set low_memory=False.
  listing = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week3/listings_with_rates.csv')


Raw MLS data contains formatting inconsistencies, missing values, and fields that need transformation before analysis. This phase prepares the dataset for reliable analytics.

## Tasks

### Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate).

In [3]:
sold[['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']].dtypes

CloseDate                   object
PurchaseContractDate        object
ListingContractDate         object
ContractStatusChangeDate    object
dtype: object

In [4]:
date_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
sold[date_cols] = sold[date_cols].apply(pd.to_datetime)
sold[['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']].dtypes

CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

In [5]:
listing[['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']].dtypes

CloseDate                   object
PurchaseContractDate        object
ListingContractDate         object
ContractStatusChangeDate    object
dtype: object

In [6]:
listing[date_cols] = listing[date_cols].apply(pd.to_datetime)
listing[['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']].dtypes

CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

### Remove unnecessary or redundant columns. Handle missing values appropriately.

In [7]:
print(sold.dtypes.to_string())

BuyerAgentAOR                           object
ListAgentAOR                            object
Flooring                                object
ViewYN                                  object
WaterfrontYN                            object
BasementYN                              object
PoolPrivateYN                           object
OriginalListPrice                      float64
ListingKey                               int64
ListAgentEmail                          object
CloseDate                       datetime64[ns]
ClosePrice                             float64
ListAgentFirstName                      object
ListAgentLastName                       object
Latitude                               float64
Longitude                              float64
UnparsedAddress                         object
PropertyType                            object
LivingArea                             float64
ListPrice                              float64
DaysOnMarket                             int64
ListOfficeNam

In [11]:
# 1. Dropped due to >90% missing (identified earlier)
sold_cols_high_missing = [
    'CoveredSpaces', 'FireplacesTotal', 'AboveGradeFinishedArea',
    'WaterfrontYN', 'BelowGradeFinishedArea', 'BasementYN', 'LotSizeDimensions',
    'TaxYear', 'BusinessType', 'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict',
    'TaxAnnualAmount', 'BuilderName', 'BuildingAreaTotal',
    'CoBuyerAgentFirstName', 'OriginatingSystemName', 'OriginatingSystemSubName',
]

# 2. Redundant — duplicated information
sold_cols_redundant_duplicate = [
    'LotSizeArea',       # same info as LotSizeSquareFeet, keep sqft as standard unit
    'LotSizeAcres',      # same info as LotSizeSquareFeet
    'latfilled',         # data quality flag, not an analytical feature
    'lonfilled',         # same
]

# 3. Metadata — >90% missing (identified earlier)
sold_cols_metadata = [
    'TaxYear', 'MiddleOrJuniorSchoolDistrict', 'ElementarySchoolDistrict',
    'BusinessType', 'TaxAnnualAmount', 'CoBuyerAgentFirstName',
    'BuilderName', 'BuildingAreaTotal', 'OriginatingSystemName', 'OriginatingSystemSubName',
]

# Drop all
cols_to_drop = (sold_cols_high_missing + sold_cols_redundant_duplicate + sold_cols_metadata)

sold_clean = sold.drop(columns=cols_to_drop)
print(f"Columns before: {sold.shape[1]}")
print(f"Columns after:  {sold_clean.shape[1]}")
print(f"Dropped:        {len(cols_to_drop)}")

Columns before: 86
Columns after:  65
Dropped:        31


In [12]:
print(listing.dtypes.to_string())

OriginalListPrice                      float64
ListingKey                               int64
ListAgentEmail                          object
CloseDate                       datetime64[ns]
ClosePrice                             float64
ListAgentFirstName                      object
ListAgentLastName                       object
Latitude                               float64
Longitude                              float64
UnparsedAddress                         object
PropertyType                            object
LivingArea                             float64
ListPrice                              float64
DaysOnMarket                             int64
ListOfficeName                          object
BuyerOfficeName                         object
CoListOfficeName                        object
ListAgentFullName                       object
CoListAgentFirstName                    object
CoListAgentLastName                     object
BuyerAgentMlsId                         object
BuyerAgentFir

In [13]:
# 1. Dropped due to >90% missing (identified earlier)
listing_cols_high_missing = [
    'CoveredSpaces', 'AboveGradeFinishedArea', 'FireplacesTotal',
    'BelowGradeFinishedArea', 'LotSizeDimensions',
    'TaxYear', 'MiddleOrJuniorSchoolDistrict', 'ElementarySchoolDistrict',
    'BusinessType', 'TaxAnnualAmount', 'CoBuyerAgentFirstName',
    'BuilderName', 'BuildingAreaTotal',
]

# 2. Redundant — duplicated information
listing_cols_redundant_duplicate = [
    'LotSizeArea',      # same info as LotSizeSquareFeet
    'LotSizeAcres',     # same info as LotSizeSquareFeet
    # .1 merge artifacts
    'PropertyType.1', 'ListAgentFirstName.1', 'ListAgentLastName.1',
    'DaysOnMarket.1', 'LivingArea.1', 'ListPrice.1',
    'Latitude.1', 'Longitude.1', 'CloseDate.1',
    'BuyerOfficeName.1', 'UnparsedAddress.1',
]

# 3. Metadata — >90% missing (identified earlier)
listing_cols_metadata = [
    'TaxYear', 'MiddleOrJuniorSchoolDistrict', 'ElementarySchoolDistrict',
    'BusinessType', 'TaxAnnualAmount', 'CoBuyerAgentFirstName',
    'BuilderName', 'BuildingAreaTotal'
]

# Drop all
cols_to_drop = (listing_cols_high_missing + listing_cols_redundant_duplicate + listing_cols_metadata)

listing_clean = listing.drop(columns=cols_to_drop)
print(f"Columns before: {listing.shape[1]}")
print(f"Columns after:  {listing_clean.shape[1]}")
print(f"Dropped:        {len(cols_to_drop)}")

Columns before: 86
Columns after:  60
Dropped:        34


### Ensure numeric fields are properly typed.

In [14]:
sold_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414184 entries, 0 to 414183
Data columns (total 65 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   BuyerAgentAOR                363289 non-null  object        
 1   ListAgentAOR                 367995 non-null  object        
 2   Flooring                     265536 non-null  object        
 3   ViewYN                       378908 non-null  object        
 4   PoolPrivateYN                378510 non-null  object        
 5   OriginalListPrice            413425 non-null  float64       
 6   ListingKey                   414184 non-null  int64         
 7   ListAgentEmail               385137 non-null  object        
 8   CloseDate                    414184 non-null  datetime64[ns]
 9   ClosePrice                   414182 non-null  float64       
 10  ListAgentFirstName           411076 non-null  object        
 11  ListAgentLastName         

In [15]:
listing_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566673 entries, 0 to 566672
Data columns (total 60 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            565861 non-null  float64       
 1   ListingKey                   566673 non-null  int64         
 2   ListAgentEmail               519638 non-null  object        
 3   CloseDate                    168206 non-null  datetime64[ns]
 4   ClosePrice                   146697 non-null  float64       
 5   ListAgentFirstName           562226 non-null  object        
 6   ListAgentLastName            566633 non-null  object        
 7   Latitude                     486232 non-null  float64       
 8   Longitude                    486232 non-null  float64       
 9   UnparsedAddress              565883 non-null  object        
 10  PropertyType                 566673 non-null  object        
 11  LivingArea                

In [16]:
listing_clean['BathroomsTotalInteger'] = listing_clean['BathroomsTotalInteger'].astype('Int64')
listing_clean['YearBuilt'] = listing_clean['YearBuilt'].astype('Int64')
sold_clean['BathroomsTotalInteger'] = sold_clean['BathroomsTotalInteger'].astype('Int64')
sold_clean['YearBuilt'] = sold_clean['YearBuilt'].astype('Int64')

### Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms.

In [17]:
# filter with numeric columns and check for negative values
listing_numeric_cols = listing_clean.select_dtypes(include=['int64', 'float64', 'Int64']).columns

listing_negative_summary = []

for col in listing_numeric_cols:
    negative_count = (listing_clean[col] < 0).sum()
    
    # Only include columns with negative values in the summary
    if negative_count > 0:
        listing_negative_summary.append({
            'column': col,
            'negative_count': negative_count,
            'min_value': listing_clean[col].min()
        })

listing_negative_summary = pd.DataFrame(listing_negative_summary)

listing_negative_summary

,column,negative_count,min_value
0,Latitude,4,-117.472493
1,Longitude,486081,-159.475987
2,DaysOnMarket,29,-58.000000
3,ParkingTotal,150,-143.000000


In [18]:
# filter with numeric columns and check for negative values
sold_numeric_cols = sold_clean.select_dtypes(include=['int64', 'float64', 'Int64']).columns

sold_negative_summary = []

for col in sold_numeric_cols:
    negative_count = (sold_clean[col] < 0).sum()
    
    # Only include columns with negative values in the summary
    if negative_count > 0:
        sold_negative_summary.append({
            'column': col,
            'negative_count': negative_count,
            'min_value': sold_clean[col].min()
        })

sold_negative_summary = pd.DataFrame(sold_negative_summary)

sold_negative_summary

,column,negative_count,min_value
0,Latitude,2,-117.472493
1,Longitude,398170,-177.646696
2,DaysOnMarket,47,-288.000000
3,ParkingTotal,97,-143.000000


In [19]:
# drop rows with negative values in 'DaysOnMarket'
listing_clean = listing_clean[listing_clean['DaysOnMarket'] >= 0].copy()
(listing_clean['DaysOnMarket'] < 0).sum()

0

In [20]:
sold_clean = sold_clean[sold_clean['DaysOnMarket'] >= 0].copy()
(sold_clean['DaysOnMarket'] < 0).sum()

0

## Data Consistency Checks

Validate the logical order of date fields: ListingContractDate should precede PurchaseContractDate, which should precede CloseDate. Create boolean flag columns to mark records that violate these rules:
- listing_after_close_flag
- purchase_after_close_flag
- negative_timeline_flag

In [21]:
listing_clean['listing_after_close_flag'] = (listing_clean['ListingContractDate'] > listing_clean['PurchaseContractDate'])
listing_clean['listing_after_close_flag'].value_counts()
listing_clean['purchase_after_close_flag'] = (listing_clean['PurchaseContractDate'] > listing_clean['CloseDate'])
listing_clean['purchase_after_close_flag'].value_counts()
listing_clean['negative_timeline_flag'] = listing_clean['listing_after_close_flag'] | listing_clean['purchase_after_close_flag']
listing_clean['negative_timeline_flag'].value_counts()

negative_timeline_flag
False    566102
True        542
Name: count, dtype: int64

In [22]:
sold_clean['listing_after_close_flag'] = (sold_clean['ListingContractDate'] > sold_clean['PurchaseContractDate'])
sold_clean['listing_after_close_flag'].value_counts()
sold_clean['purchase_after_close_flag'] = (sold_clean['PurchaseContractDate'] > sold_clean['CloseDate'])
sold_clean['purchase_after_close_flag'].value_counts()
sold_clean['negative_timeline_flag'] = sold_clean['listing_after_close_flag'] | sold_clean['purchase_after_close_flag']
sold_clean['negative_timeline_flag'].value_counts()

negative_timeline_flag
False    413627
True        510
Name: count, dtype: int64

# Geographic Data Checks

- Flag records with missing coordinates (Latitude or Longitude is null)
- Flag Latitude = 0 or Longitude = 0 (sentinel null values)
- Flag Longitude > 0 errors (California coordinates should be negative)
- Flag out-of-state or implausible coordinates

In [23]:
# either latitude or longitude is missing
listing_clean['missing_coordinates_flag'] = listing_clean['Latitude'].isna() | listing_clean['Longitude'].isna()
listing_clean['missing_coordinates_flag'].value_counts()

missing_coordinates_flag
False    486204
True      80440
Name: count, dtype: int64

In [24]:
# either latitude or longitude is missing
sold_clean['missing_coordinates_flag'] = sold_clean['Latitude'].isna() | sold_clean['Longitude'].isna()
sold_clean['missing_coordinates_flag'].value_counts()

missing_coordinates_flag
False    398181
True      15956
Name: count, dtype: int64

In [25]:
# either latitude or longitude == 0
listing_clean['zero_coordinates_flag'] = (listing_clean['Latitude'] == 0) | (listing_clean['Longitude'] == 0)
listing_clean['zero_coordinates_flag'].value_counts()

zero_coordinates_flag
False    566579
True         65
Name: count, dtype: int64

In [26]:
# either latitude or longitude == 0
sold_clean['zero_coordinates_flag'] = (sold_clean['Latitude'] == 0) | (sold_clean['Longitude'] == 0)
sold_clean['zero_coordinates_flag'].value_counts()

zero_coordinates_flag
False    414110
True         27
Name: count, dtype: int64

In [27]:
# longitude > 0 (should be negative in the CA)
listing_clean['invalid_longitude_ca_flag'] = (listing_clean['StateOrProvince'] == 'CA') & (listing_clean['Longitude'] > 0)
listing_clean['invalid_longitude_ca_flag'].value_counts()

invalid_longitude_ca_flag
False    566596
True         48
Name: count, dtype: int64

In [28]:
# longitude > 0 (should be negative in the CA)
sold_clean['invalid_longitude_ca_flag'] = (sold_clean['StateOrProvince'] == 'CA') & (sold_clean['Longitude'] > 0)
sold_clean['invalid_longitude_ca_flag'].value_counts()

invalid_longitude_ca_flag
False    414106
True         31
Name: count, dtype: int64

In [29]:
listing_clean['out_of_ca_flag'] = (
    (listing_clean['Latitude'] < 32) |
    (listing_clean['Latitude'] > 42) |
    (listing_clean['Longitude'] < -124) |
    (listing_clean['Longitude'] > -114)
)
listing_clean['out_of_ca_flag'].value_counts()

out_of_ca_flag
False    566311
True        333
Name: count, dtype: int64

In [30]:
sold_clean['out_of_ca_flag'] = (
    (sold_clean['Latitude'] < 32) |
    (sold_clean['Latitude'] > 42) |
    (sold_clean['Longitude'] < -124) |
    (sold_clean['Longitude'] > -114)
)
sold_clean['out_of_ca_flag'].value_counts()

out_of_ca_flag
False    414039
True         98
Name: count, dtype: int64

In [31]:
listing_clean['implausible_coordinates_flag'] = (
    (listing_clean['Latitude'] < -90) |
    (listing_clean['Latitude'] > 90) |
    (listing_clean['Longitude'] < -180) |
    (listing_clean['Longitude'] > 180)
)
listing_clean['implausible_coordinates_flag'].value_counts()

implausible_coordinates_flag
False    566640
True          4
Name: count, dtype: int64

In [32]:
sold_clean['implausible_coordinates_flag'] = (
    (sold_clean['Latitude'] < -90) |
    (sold_clean['Latitude'] > 90) |
    (sold_clean['Longitude'] < -180) |
    (sold_clean['Longitude'] > 180)
)
sold_clean['implausible_coordinates_flag'].value_counts()

implausible_coordinates_flag
False    414135
True          2
Name: count, dtype: int64

# Removing entries that contain the above inconsistencies

In [33]:
listing_clean.shape

(566644, 68)

In [34]:
listing_clean_updated = listing_clean.drop(listing_clean[(listing_clean['negative_timeline_flag'] == True) |
                                                         (listing_clean['missing_coordinates_flag'] == True) |
                                                         (listing_clean['zero_coordinates_flag'] == True) |
                                                         (listing_clean['invalid_longitude_ca_flag'] == True) |
                                                         (listing_clean['out_of_ca_flag'] == True) |
                                                         (listing_clean['implausible_coordinates_flag'] == True)].index)
listing_clean_updated.shape

(485492, 68)

In [35]:
sold_clean.shape

(414137, 73)

In [36]:
sold_clean_updated = sold_clean.drop(sold_clean[(sold_clean['negative_timeline_flag'] == True) |
                                                   (sold_clean['missing_coordinates_flag'] == True) |
                                                   (sold_clean['zero_coordinates_flag'] == True) |
                                                   (sold_clean['invalid_longitude_ca_flag'] == True) |
                                                   (sold_clean['out_of_ca_flag'] == True) |
                                                   (sold_clean['implausible_coordinates_flag'] == True)].index)
sold_clean_updated.shape

(397657, 73)

In [37]:
listing_clean_updated.to_csv('/Users/olee1232/Documents/IDXWork/Week4-5/listing_clean.csv', index=False) 
sold_clean_updated.to_csv('/Users/olee1232/Documents/IDXWork/Week4-5/sold_clean.csv', index=False) 